# RAG Mémoire — ADS CESI 2025-2026
## Notebook Kaggle — Expériences RAG Bilingues

Ce notebook exécute les protocoles expérimentaux A, B, C sur GPU Kaggle (T4/P100).

**Architecture** :
- Retrieval : BM25 (Pyserini/Lucene), Dense (FAISS + multilingual-e5-large / DPR), Hybride
- Reranking : ColBERT (RAGatouille)
- Génération : mT5-large (constant)
- Évaluation : EM, F1, Recall@k, MRR, nDCG, Faithfulness (RAGAS)

**Paramètres Kaggle** : Activer GPU dans *Settings → Accelerator → GPU T4 x1*

## Cellule 1 — Installation des dépendances
> ⏱ ~5-8 minutes (Java + pyserini + faiss-gpu)

In [ ]:
import os
import sys

# ── Java requis par Pyserini (Lucene) ─────────────────────────────────────────
os.system("apt-get install -y -q default-jdk-headless 2>/dev/null")

# ── Vérification GPU ──────────────────────────────────────────────────────────
import subprocess
gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print("GPU disponible :")
    print(gpu_info.stdout[:500])
    USE_GPU = True
    FAISS_PKG = "faiss-gpu"
else:
    print("Pas de GPU — mode CPU (plus lent)")
    USE_GPU = False
    FAISS_PKG = "faiss-cpu"

# ── Installation des packages ─────────────────────────────────────────────────
packages = [
    "pyserini>=0.22.0",
    f"{FAISS_PKG}>=1.7.4",
    "transformers>=4.40.0",
    "accelerate>=0.29.0",
    "ragatouille>=0.0.8",
    "sentence-transformers>=3.0.0",
    "ragas>=0.1.7",
    "evaluate>=0.4.1",
    "scipy>=1.13.0",
    "mlxtend>=0.23.0",
    "datasets>=2.19.0",
    "jsonlines>=4.0.0",
    "pytrec_eval>=0.5",
    "pyyaml>=6.0.1",
    "tqdm>=4.66.0",
]

pip_cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
result = subprocess.run(pip_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("ERREUR pip :", result.stderr[-1000:])
else:
    print("✓ Packages installés")

## Cellule 2 — Cloner le dépôt et configurer les chemins

In [ ]:
import os
import sys
from pathlib import Path

# ── Chemins Kaggle ─────────────────────────────────────────────────────────────
KAGGLE_WORKING = Path("/kaggle/working")
REPO_DIR = KAGGLE_WORKING / "rag-memoire"

# ── Cloner le repo depuis GitHub (remplacer l'URL par la vôtre) ───────────────
GITHUB_REPO_URL = "https://github.com/VOTRE_USERNAME/rag-memoire.git"  # ← À MODIFIER

if not REPO_DIR.exists():
    print(f"Clonage de {GITHUB_REPO_URL}...")
    result = subprocess.run(
        ["git", "clone", "--depth=1", GITHUB_REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("ERREUR git clone :", result.stderr)
        print("→ Le code source sera injecté inline dans les cellules suivantes.")
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        USE_INLINE_CODE = True
    else:
        print(f"✓ Repo cloné → {REPO_DIR}")
        USE_INLINE_CODE = False
else:
    print(f"✓ Repo déjà présent : {REPO_DIR}")
    USE_INLINE_CODE = False

# ── Ajouter au PYTHONPATH ─────────────────────────────────────────────────────
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Créer les répertoires de travail ─────────────────────────────────────────
for d in ["data/raw", "data/processed", "data/splits",
          "indexes/bm25", "indexes/dense", "indexes/hybrid",
          "runs/A_ablation", "runs/B_context_noise", "runs/C_factuality",
          "reports/logs", "reports/figures", "reports/tables",
          "configs"]:
    (REPO_DIR / d).mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
print(f"✓ Répertoire de travail : {os.getcwd()}")

## Cellule 3 — Injection des fichiers de configuration YAML
> Exécuter uniquement si le repo n'a pas pu être cloné

In [ ]:
# ── Configs YAML (nécessaire si USE_INLINE_CODE = True ou configs absentes) ──

datasets_yaml = """
datasets:
  fquad2:
    name: "FQuAD 2.0"
    lang: "fr"
    hf_id: "illuin-technology/fquad2"
    splits: ["train", "validation"]
    type: "qa_unanswerable"
    protocols: ["A", "B", "C"]
  piaf:
    name: "PIAF v1.1"
    lang: "fr"
    hf_id: "illuin-technology/piaf"
    splits: ["train"]
    type: "qa"
    protocols: ["A"]
  kilt_nq:
    name: "KILT Natural Questions"
    lang: "en"
    hf_id: "kilt_tasks"
    hf_subset: "nq"
    splits: ["train", "validation"]
    type: "qa_open_domain"
    protocols: ["A", "B"]
checksums_file: "checksums.txt"
data_dir: "data/"
raw_dir: "data/raw/"
processed_dir: "data/processed/"
splits_dir: "data/splits/"
"""

retrievers_yaml = """
bm25:
  index_dir: "indexes/bm25/"
  k1: 1.0
  b: 0.6

dense:
  index_dir: "indexes/dense/"
  models:
    fr: "intfloat/multilingual-e5-large"
    en: "facebook/dpr-ctx_encoder-single-nq-base"
  batch_size: 64

hybrid:
  alpha: 0.5
  alpha_grid: [0.3, 0.5, 0.7]
"""

generators_yaml = """
generator:
  model_name: "google/mt5-large"
  max_new_tokens: 128
  num_beams: 4
  early_stopping: true
  no_repeat_ngram_size: 3
  seed: 42
"""

prompts_yaml = """
standard:
  template: |
    Context:\n{context}\n\nQuestion: {question}\nAnswer:

citations:
  template: |
    {numbered_context}\n\nQuestion: {question}\nAnswer using [N] citations:\nAnswer:

baseline:
  template: |
    Question: {question}\nAnswer:
"""

configs_dir = REPO_DIR / "configs"
configs_dir.mkdir(exist_ok=True)

(configs_dir / "datasets.yaml").write_text(datasets_yaml.strip(), encoding="utf-8")
(configs_dir / "retrievers.yaml").write_text(retrievers_yaml.strip(), encoding="utf-8")
(configs_dir / "generators.yaml").write_text(generators_yaml.strip(), encoding="utf-8")
(configs_dir / "prompts.yaml").write_text(prompts_yaml.strip(), encoding="utf-8")

print("✓ Configs YAML écrits")

## Cellule 4 — Paramètres globaux de l'expérience

In [ ]:
import random
import numpy as np
import torch
import logging

# ── Seeds reproductibilité ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s — %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(str(REPO_DIR / "reports/logs/experiments.log"), encoding="utf-8"),
    ]
)
logger = logging.getLogger("kaggle_rag")

# ── Paramètres Kaggle ─────────────────────────────────────────────────────────
# Réduire MAX_SAMPLES pour les tests rapides (None = tout le dataset)
MAX_SAMPLES = 200          # ← Réduire pour un test rapide, None pour tout
PROTOCOL = "A"             # ← "A", "B", "C", ou "all"
LANG = "fr"                # ← "fr", "en", ou "both"
CHUNK_SIZE = 256           # taille des chunks en tokens
CHUNK_OVERLAP = 50         # chevauchement

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Protocole : {PROTOCOL} | Langue : {LANG} | Max samples : {MAX_SAMPLES}")
if DEVICE == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cellule 5 — Chargement des datasets

In [ ]:
from datasets import load_dataset
from tqdm.notebook import tqdm

def load_fquad2(split="validation", max_samples=None):
    """Charge FQuAD 2.0 depuis HuggingFace."""
    logger.info("Chargement FQuAD 2.0 — split: %s", split)
    ds = load_dataset("illuin-technology/fquad2", split=split, trust_remote_code=True)
    samples = []
    for item in tqdm(ds, desc="FQuAD 2.0"):
        answers = item.get("answers", {})
        answer_texts = answers.get("text", []) if isinstance(answers, dict) else []
        samples.append({
            "id": item.get("id", ""),
            "question": item["question"],
            "context": item.get("context", ""),
            "answers": answer_texts,
            "is_impossible": item.get("is_impossible", False),
            "lang": "fr",
            "source": "fquad2",
        })
        if max_samples and len(samples) >= max_samples:
            break
    logger.info("%d exemples FQuAD 2.0 chargés", len(samples))
    return samples


def load_kilt_nq(split="validation", max_samples=None):
    """Charge KILT Natural Questions."""
    logger.info("Chargement KILT NQ — split: %s", split)
    ds = load_dataset("kilt_tasks", "nq", split=split, trust_remote_code=True)
    samples = []
    for item in tqdm(ds, desc="KILT NQ"):
        output = item.get("output", [])
        answers = [o["answer"] for o in output if o.get("answer")]
        samples.append({
            "id": item["id"],
            "question": item["input"],
            "context": "",
            "answers": answers,
            "is_impossible": len(answers) == 0,
            "lang": "en",
            "source": "kilt_nq",
        })
        if max_samples and len(samples) >= max_samples:
            break
    logger.info("%d exemples KILT NQ chargés", len(samples))
    return samples


# ── Chargement selon la langue choisie ────────────────────────────────────────
all_samples = {}
langs = ["fr", "en"] if LANG == "both" else [LANG]

for lang in langs:
    if lang == "fr":
        all_samples["fr"] = load_fquad2(split="validation", max_samples=MAX_SAMPLES)
        # Corpus de passages : utiliser les contextes FQuAD comme passages
        corpus_fr = [
            {"id": s["id"], "text": s["context"], "lang": "fr"}
            for s in all_samples["fr"]
            if s.get("context")
        ]
        print(f"FR : {len(all_samples['fr'])} questions, {len(corpus_fr)} passages corpus")
    elif lang == "en":
        all_samples["en"] = load_kilt_nq(split="validation", max_samples=MAX_SAMPLES)
        corpus_en = [
            {"id": s["id"], "text": s["context"], "lang": "en"}
            for s in all_samples["en"]
            if s.get("context")
        ]
        print(f"EN : {len(all_samples['en'])} questions, {len(corpus_en)} passages corpus")

## Cellule 6 — Normalisation et chunking du corpus

In [ ]:
import unicodedata
import re
from dataclasses import dataclass, field
from typing import Any


# ── Normalizer ────────────────────────────────────────────────────────────────
class TextNormalizer:
    """Normalisation Unicode et nettoyage du texte."""

    def normalize(self, text: str) -> str:
        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f]", " ", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def normalize_document(self, doc: dict) -> dict:
        doc["text"] = self.normalize(doc.get("text", ""))
        return doc


# ── Passage dataclass ─────────────────────────────────────────────────────────
@dataclass
class Passage:
    id: str
    text: str
    metadata: dict[str, Any] = field(default_factory=dict)


# ── Chunker ───────────────────────────────────────────────────────────────────
class DocumentChunker:
    """Découpe les documents en passages avec chevauchement."""

    def __init__(
        self,
        chunk_size: int = 256,
        overlap: int = 50,
        model_name: str = "intfloat/multilingual-e5-large",
    ) -> None:
        self.chunk_size = chunk_size
        self.overlap = overlap
        try:
            from transformers import AutoTokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        except Exception:
            self.tokenizer = None
            logger.warning("Tokenizer non disponible, chunking par caractères")

    def _chunk_by_tokens(self, text: str, doc_id: str, lang: str) -> list[Passage]:
        ids = self.tokenizer.encode(text, add_special_tokens=False)
        passages = []
        step = self.chunk_size - self.overlap
        for i, start in enumerate(range(0, len(ids), step)):
            chunk_ids = ids[start: start + self.chunk_size]
            chunk_text = self.tokenizer.decode(chunk_ids, skip_special_tokens=True)
            if len(chunk_text.strip()) < 20:
                continue
            passages.append(Passage(
                id=f"{doc_id}_chunk_{i:04d}",
                text=chunk_text.strip(),
                metadata={"source_doc_id": doc_id, "chunk_index": i, "lang": lang},
            ))
        return passages

    def _chunk_by_chars(self, text: str, doc_id: str, lang: str) -> list[Passage]:
        char_size = self.chunk_size * 4
        char_overlap = self.overlap * 4
        step = char_size - char_overlap
        passages = []
        for i, start in enumerate(range(0, len(text), step)):
            chunk = text[start: start + char_size].strip()
            if len(chunk) < 20:
                continue
            passages.append(Passage(
                id=f"{doc_id}_chunk_{i:04d}",
                text=chunk,
                metadata={"source_doc_id": doc_id, "chunk_index": i, "lang": lang},
            ))
        return passages

    def chunk_document(self, doc: dict) -> list[Passage]:
        text = doc.get("text", "")
        doc_id = doc.get("id", "unknown")
        lang = doc.get("lang", "fr")
        if not text.strip():
            return []
        if self.tokenizer:
            return self._chunk_by_tokens(text, doc_id, lang)
        return self._chunk_by_chars(text, doc_id, lang)

    def chunk_corpus(self, documents: list[dict]) -> list[Passage]:
        normalizer = TextNormalizer()
        all_passages = []
        for doc in tqdm(documents, desc="Chunking"):
            doc = normalizer.normalize_document(doc)
            all_passages.extend(self.chunk_document(doc))
        logger.info("%d passages générés depuis %d documents", len(all_passages), len(documents))
        return all_passages


# ── Chunking du corpus ────────────────────────────────────────────────────────
chunker = DocumentChunker(chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

all_passages = {}
for lang in langs:
    corpus = corpus_fr if lang == "fr" else corpus_en
    passages = chunker.chunk_corpus(corpus)
    all_passages[lang] = passages
    print(f"[{lang.upper()}] {len(passages)} passages générés")

## Cellule 7 — Construction des index BM25 (Pyserini)

In [ ]:
import json
import subprocess
from pathlib import Path


def prepare_corpus_for_pyserini(passages: list, output_dir: str) -> None:
    """Prépare le corpus au format JsonCollection pour Pyserini."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    batch_size = 10000
    for batch_idx in range(0, len(passages), batch_size):
        batch = passages[batch_idx: batch_idx + batch_size]
        filename = output_path / f"passages_{batch_idx:08d}.jsonl"
        with open(filename, "w", encoding="utf-8") as f:
            for p in batch:
                doc = {"id": p.id, "contents": p.text}
                doc.update(p.metadata)
                f.write(json.dumps(doc, ensure_ascii=False) + "\n")
    logger.info("Corpus Pyserini prêt : %d passages → %s", len(passages), output_dir)


def build_bm25_index(processed_dir: str, index_dir: str, threads: int = 4) -> bool:
    """Construit l'index BM25 via Pyserini/Lucene."""
    logger.info("Construction index BM25 → %s", index_dir)
    Path(index_dir).mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "-m", "pyserini.index.lucene",
        "--collection", "JsonCollection",
        "--input", processed_dir,
        "--index", index_dir,
        "--generator", "DefaultLuceneDocumentGenerator",
        "--threads", str(threads),
        "--storePositions",
        "--storeDocvectors",
        "--storeRaw",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_DIR))
    if result.returncode != 0:
        logger.error("Erreur BM25 : %s", result.stderr[-500:])
        return False
    logger.info("Index BM25 construit avec succès")
    return True


# ── Construire les index BM25 par langue ─────────────────────────────────────
bm25_index_dirs = {}
for lang in langs:
    processed_dir = str(REPO_DIR / f"data/processed/{lang}")
    index_dir = str(REPO_DIR / f"indexes/bm25/{lang}")
    bm25_index_dirs[lang] = index_dir

    prepare_corpus_for_pyserini(all_passages[lang], processed_dir)
    success = build_bm25_index(processed_dir, index_dir)
    if success:
        print(f"[{lang.upper()}] ✓ Index BM25 construit")
    else:
        print(f"[{lang.upper()}] ✗ Échec BM25 — vérifier Java et Pyserini")

## Cellule 8 — Construction des index Dense (FAISS)

In [ ]:
import pickle
import numpy as np
import torch
import faiss
from transformers import AutoModel, AutoTokenizer


DENSE_MODELS = {
    "fr": "intfloat/multilingual-e5-large",
    "en": "facebook/dpr-question_encoder-single-nq-base",
}
DENSE_QUERY_MODELS = {
    "fr": "intfloat/multilingual-e5-large",
    "en": "facebook/dpr-question_encoder-single-nq-base",
}


def encode_passages_batch(
    texts: list,
    tokenizer,
    model,
    device: str,
    batch_size: int = 32,
    is_query: bool = False,
    lang: str = "fr",
) -> np.ndarray:
    """Encode des textes en vecteurs denses avec prefix E5 si nécessaire."""
    if "e5" in getattr(model, "name_or_path", "").lower() or lang == "fr":
        prefix = "query: " if is_query else "passage: "
        texts = [prefix + t for t in texts]

    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encodage", leave=False):
        batch = texts[i: i + batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            emb = outputs.pooler_output
        else:
            emb = outputs.last_hidden_state[:, 0, :]

        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)


def build_faiss_index(
    passages: list,
    lang: str,
    index_dir: str,
    device: str = DEVICE,
    batch_size: int = 32,
) -> None:
    """Construit et sauvegarde l'index FAISS pour une langue."""
    model_name = DENSE_MODELS[lang]
    index_path = Path(index_dir)
    index_path.mkdir(parents=True, exist_ok=True)

    logger.info("Chargement modèle dense : %s", model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    texts = [p.text for p in passages]
    passage_ids = [p.id for p in passages]

    logger.info("Encodage de %d passages...", len(texts))
    embeddings = encode_passages_batch(
        texts, tokenizer, model, device, batch_size=batch_size, is_query=False, lang=lang
    ).astype(np.float32)

    dim = embeddings.shape[1]
    if len(passages) > 50000:
        quantizer = faiss.IndexFlatIP(dim)
        nlist = min(4096, len(passages) // 10)
        index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        index.train(embeddings)
    else:
        index = faiss.IndexFlatIP(dim)

    # GPU FAISS si disponible
    if device == "cuda" and FAISS_PKG == "faiss-gpu":
        try:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
            logger.info("Index FAISS sur GPU")
        except Exception:
            logger.warning("GPU FAISS non disponible, utilisation CPU")

    index.add(embeddings)
    logger.info("Index FAISS : %d vecteurs dim %d", index.ntotal, dim)

    # Sauvegarder sur CPU
    try:
        cpu_index = faiss.index_gpu_to_cpu(index)
    except Exception:
        cpu_index = index

    faiss.write_index(cpu_index, str(index_path / f"index_{lang}.faiss"))
    with open(index_path / f"passage_ids_{lang}.pkl", "wb") as f:
        pickle.dump(passage_ids, f)
    with open(index_path / f"passage_texts_{lang}.pkl", "wb") as f:
        pickle.dump(texts, f)
    logger.info("Index sauvegardé → %s", index_path)

    del model
    torch.cuda.empty_cache() if device == "cuda" else None


# ── Construire les index FAISS par langue ────────────────────────────────────
dense_index_dir = str(REPO_DIR / "indexes/dense")
for lang in langs:
    print(f"\n[{lang.upper()}] Construction index FAISS...")
    build_faiss_index(all_passages[lang], lang, dense_index_dir)
    print(f"[{lang.upper()}] ✓ Index FAISS construit")

## Cellule 9 — Classes Retriever

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class RetrievedPassage:
    rank: int
    passage_id: str
    text: str
    bm25_score: float = 0.0
    dense_score: float = 0.0
    hybrid_score: float = 0.0
    rerank_score: float = 0.0
    metadata: dict = field(default_factory=dict)

    def to_dict(self):
        return {
            "rank": self.rank, "passage_id": self.passage_id, "text": self.text,
            "bm25_score": self.bm25_score, "dense_score": self.dense_score,
            "hybrid_score": self.hybrid_score, "rerank_score": self.rerank_score,
            "metadata": self.metadata,
        }


# ── BM25 Retriever ─────────────────────────────────────────────────────────────
class BM25Retriever:
    def __init__(self, index_dir: str, k1: float = 1.0, b: float = 0.6):
        self.index_dir = index_dir
        self.k1 = k1
        self.b = b
        self._searcher = None

    def _load_searcher(self):
        from pyserini.search.lucene import LuceneSearcher
        self._searcher = LuceneSearcher(self.index_dir)
        self._searcher.set_bm25(self.k1, self.b)
        logger.info("BM25 chargé : %s", self.index_dir)

    def retrieve(self, query: str, k: int = 10) -> list:
        if self._searcher is None:
            self._load_searcher()
        hits = self._searcher.search(query, k=k)
        passages = []
        for rank, hit in enumerate(hits, start=1):
            try:
                doc = json.loads(hit.raw)
                text = doc.get("contents", "")
            except Exception:
                text = hit.raw or ""
            passages.append(RetrievedPassage(
                rank=rank, passage_id=hit.docid, text=text, bm25_score=float(hit.score)
            ))
        return passages


# ── Dense Retriever ────────────────────────────────────────────────────────────
class DenseRetriever:
    def __init__(self, index_dir: str, lang: str, device: str = DEVICE, batch_size: int = 32):
        self.index_dir = Path(index_dir)
        self.lang = lang
        self.device = device
        self.batch_size = batch_size
        self.model_name = DENSE_QUERY_MODELS[lang]
        self._model = None
        self._tokenizer = None
        self._index = None
        self._passage_ids = []
        self._passage_texts = []

    def _load(self):
        import faiss
        self._tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self._model = AutoModel.from_pretrained(self.model_name).to(self.device)
        self._model.eval()
        idx_path = self.index_dir / f"index_{self.lang}.faiss"
        self._index = faiss.read_index(str(idx_path))
        with open(self.index_dir / f"passage_ids_{self.lang}.pkl", "rb") as f:
            self._passage_ids = pickle.load(f)
        with open(self.index_dir / f"passage_texts_{self.lang}.pkl", "rb") as f:
            self._passage_texts = pickle.load(f)
        logger.info("Dense retriever chargé : %d passages", len(self._passage_ids))

    def retrieve(self, query: str, k: int = 10) -> list:
        if self._index is None:
            self._load()
        emb = encode_passages_batch(
            [query], self._tokenizer, self._model, self.device,
            batch_size=1, is_query=True, lang=self.lang
        ).astype(np.float32)
        scores, indices = self._index.search(emb, k)
        passages = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
            if idx < 0:
                continue
            passages.append(RetrievedPassage(
                rank=rank, passage_id=self._passage_ids[idx],
                text=self._passage_texts[idx], dense_score=float(score)
            ))
        return passages


# ── Hybrid Retriever ───────────────────────────────────────────────────────────
class HybridRetriever:
    def __init__(self, bm25: BM25Retriever, dense: DenseRetriever, alpha: float = 0.5):
        self.bm25 = bm25
        self.dense = dense
        self.alpha = alpha

    @staticmethod
    def _normalize(passages, score_type):
        scores = {p.passage_id: (p.bm25_score if score_type == "bm25" else p.dense_score)
                  for p in passages}
        if not scores:
            return {}
        lo, hi = min(scores.values()), max(scores.values())
        denom = hi - lo if hi != lo else 1.0
        return {pid: (s - lo) / denom for pid, s in scores.items()}

    def retrieve(self, query: str, k: int = 10) -> list:
        initial_k = max(k * 3, 50)
        bm25_res = self.bm25.retrieve(query, k=initial_k)
        dense_res = self.dense.retrieve(query, k=initial_k)
        b_norm = self._normalize(bm25_res, "bm25")
        d_norm = self._normalize(dense_res, "dense")
        all_ids = set(b_norm) | set(d_norm)
        idx = {p.passage_id: p for p in bm25_res + dense_res}
        fused = []
        for pid in all_ids:
            b = b_norm.get(pid, 0.0)
            d = d_norm.get(pid, 0.0)
            h = self.alpha * d + (1 - self.alpha) * b
            p = idx[pid]
            fused.append(RetrievedPassage(
                rank=0, passage_id=pid, text=p.text,
                bm25_score=p.bm25_score, dense_score=p.dense_score, hybrid_score=h,
            ))
        fused.sort(key=lambda x: x.hybrid_score, reverse=True)
        for rank, p in enumerate(fused[:k], start=1):
            p.rank = rank
        return fused[:k]


print("✓ Classes Retriever définies")

## Cellule 10 — Prompt Formatter & Context Builder

In [ ]:
class PromptFormatter:
    """Formatage des prompts standard, citations, et baseline."""

    def format_standard(self, question: str, passages: list) -> str:
        context = "\n\n".join(passages)
        return f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

    def format_citations(self, question: str, passages: list) -> str:
        numbered = "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages))
        return f"{numbered}\n\nQuestion: {question}\nAnswer using [N] citations:\nAnswer:"

    def format_baseline(self, question: str) -> str:
        return f"Question: {question}\nAnswer:"

    def format(self, question: str, passages: list, fmt: str = "standard") -> str:
        if fmt == "citations":
            return self.format_citations(question, passages)
        return self.format_standard(question, passages)


class ContextBuilder:
    """Construction et réorganisation du contexte."""

    def build(self, passages: list, gold_passage_id: str, position: str = "first") -> list:
        """Réorganise les passages selon la position du passage gold."""
        if not gold_passage_id:
            return passages
        gold = [p for p in passages if p.passage_id == gold_passage_id]
        others = [p for p in passages if p.passage_id != gold_passage_id]
        if not gold:
            return passages
        g = gold[0]
        if position == "first":
            return [g] + others
        elif position == "last":
            return others + [g]
        elif position == "middle":
            mid = len(others) // 2
            return others[:mid] + [g] + others[mid:]
        return passages

    def build_with_distractors(
        self,
        gold: RetrievedPassage,
        distractors: list,
        k_total: int = 10,
        distractor_ratio: float = 0.4,
        position: str = "first",
    ) -> list:
        """Construit le contexte avec un ratio contrôlé de distracteurs."""
        n_distractors = min(int(k_total * distractor_ratio), len(distractors))
        selected_distractors = distractors[:n_distractors]
        n_gold = 1
        n_pad = k_total - n_distractors - n_gold
        n_pad = max(0, n_pad)
        padding = distractors[n_distractors: n_distractors + n_pad]
        others = selected_distractors + padding
        if position == "first":
            return [gold] + others
        elif position == "last":
            return others + [gold]
        elif position == "middle":
            mid = len(others) // 2
            return others[:mid] + [gold] + others[mid:]
        return [gold] + others


print("✓ PromptFormatter et ContextBuilder définis")

## Cellule 11 — Générateur mT5-large

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as HFTokenizer


class RAGGenerator:
    """Wrapper mT5-large pour la génération de réponses (constant dans toutes les expériences)."""

    MODEL_NAME = "google/mt5-large"
    GENERATION_PARAMS = {
        "max_new_tokens": 128,
        "num_beams": 4,
        "early_stopping": True,
        "no_repeat_ngram_size": 3,
    }

    def __init__(self, device: str = DEVICE, seed: int = 42):
        self.device = device
        self.seed = seed
        self.dtype = torch.float16 if device == "cuda" else torch.float32
        self._model = None
        self._tokenizer = None

    def _load(self):
        logger.info("Chargement mT5-large sur %s", self.device)
        self._tokenizer = HFTokenizer.from_pretrained(self.MODEL_NAME)
        self._model = AutoModelForSeq2SeqLM.from_pretrained(
            self.MODEL_NAME, torch_dtype=self.dtype
        ).to(self.device)
        self._model.eval()
        logger.info("mT5-large chargé")

    def generate(self, prompt: str) -> str:
        if self._model is None:
            self._load()
        inputs = self._tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=1024
        ).to(self.device)
        with torch.no_grad():
            out = self._model.generate(**inputs, **self.GENERATION_PARAMS)
        return self._tokenizer.decode(out[0], skip_special_tokens=True).strip()

    def generate_batch(self, prompts: list, batch_size: int = 4) -> list:
        if self._model is None:
            self._load()
        answers = []
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i: i + batch_size]
            inputs = self._tokenizer(
                batch, return_tensors="pt", padding=True, truncation=True, max_length=1024
            ).to(self.device)
            with torch.no_grad():
                out = self._model.generate(**inputs, **self.GENERATION_PARAMS)
            for ids in out:
                answers.append(self._tokenizer.decode(ids, skip_special_tokens=True).strip())
        return answers


# ── Initialiser le générateur (lazy loading — modèle chargé à la première inférence) ──
generator = RAGGenerator(device=DEVICE)
print(f"✓ RAGGenerator initialisé (device={DEVICE}, lazy loading)")

## Cellule 12 — Métriques d'évaluation

In [ ]:
import string
import math
import re
from collections import Counter


# ── QA Metrics ─────────────────────────────────────────────────────────────────
class QAMetrics:
    FR_ARTICLES = {"le", "la", "les", "un", "une", "des", "l", "d"}
    EN_ARTICLES = {"the", "a", "an"}

    def _normalize(self, text: str, lang: str = "fr") -> list:
        text = text.lower()
        text = text.translate(str.maketrans("", "", string.punctuation))
        tokens = text.split()
        stopwords = self.FR_ARTICLES if lang == "fr" else self.EN_ARTICLES
        return [t for t in tokens if t not in stopwords]

    def exact_match(self, pred: str, gold_list: list, lang: str = "fr") -> int:
        pred_norm = " ".join(self._normalize(pred, lang))
        for gold in gold_list:
            if pred_norm == " ".join(self._normalize(gold, lang)):
                return 1
        return 0

    def token_f1(self, pred: str, gold_list: list, lang: str = "fr") -> float:
        pred_tokens = self._normalize(pred, lang)
        best_f1 = 0.0
        for gold in gold_list:
            gold_tokens = self._normalize(gold, lang)
            common = Counter(pred_tokens) & Counter(gold_tokens)
            n_common = sum(common.values())
            if n_common == 0:
                continue
            precision = n_common / len(pred_tokens) if pred_tokens else 0
            recall = n_common / len(gold_tokens) if gold_tokens else 0
            f1 = 2 * precision * recall / (precision + recall)
            best_f1 = max(best_f1, f1)
        return best_f1

    def compute_batch(self, preds: list, gold_lists: list, lang: str = "fr") -> dict:
        ems = [self.exact_match(p, g, lang) for p, g in zip(preds, gold_lists)]
        f1s = [self.token_f1(p, g, lang) for p, g in zip(preds, gold_lists)]
        return {"em": sum(ems) / len(ems), "f1": sum(f1s) / len(f1s)}


# ── Retrieval Metrics ─────────────────────────────────────────────────────────
class RetrievalMetrics:

    def recall_at_k(self, retrieved: list, relevant: set, k: int = 10) -> float:
        top_k = retrieved[:k]
        hits = sum(1 for r in top_k if r in relevant)
        return hits / len(relevant) if relevant else 0.0

    def mrr(self, retrieved: list, relevant: set) -> float:
        for rank, r in enumerate(retrieved, start=1):
            if r in relevant:
                return 1.0 / rank
        return 0.0

    def ndcg_at_k(self, retrieved: list, relevance: dict, k: int = 10) -> float:
        dcg = sum(
            relevance.get(r, 0) / math.log2(rank + 1)
            for rank, r in enumerate(retrieved[:k], start=1)
        )
        ideal = sorted(relevance.values(), reverse=True)[:k]
        idcg = sum(rel / math.log2(rank + 1) for rank, rel in enumerate(ideal, start=1))
        return dcg / idcg if idcg > 0 else 0.0


# ── Statistical Tests ─────────────────────────────────────────────────────────
class StatisticalTests:

    def permutation_test(self, a: list, b: list, n_permutations: int = 2000) -> dict:
        a, b = np.array(a), np.array(b)
        obs_diff = np.mean(b) - np.mean(a)
        combined = np.concatenate([a, b])
        n = len(a)
        count = 0
        rng = np.random.default_rng(SEED)
        for _ in range(n_permutations):
            perm = rng.permutation(combined)
            diff = np.mean(perm[n:]) - np.mean(perm[:n])
            if abs(diff) >= abs(obs_diff):
                count += 1
        p_val = count / n_permutations
        stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        return {"observed_diff": float(obs_diff), "p_value": float(p_val), "stars": stars}

    def bootstrap_ci(self, scores: list, n_bootstrap: int = 1000, alpha: float = 0.05) -> dict:
        scores = np.array(scores)
        rng = np.random.default_rng(SEED)
        means = [np.mean(rng.choice(scores, size=len(scores), replace=True)) for _ in range(n_bootstrap)]
        lo, hi = np.percentile(means, 100 * alpha / 2), np.percentile(means, 100 * (1 - alpha / 2))
        return {"mean": float(np.mean(scores)), "ci_lower": float(lo), "ci_upper": float(hi)}

    def wilcoxon_test(self, a: list, b: list) -> dict:
        from scipy.stats import wilcoxon
        stat, p_val = wilcoxon(a, b)
        stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        return {"statistic": float(stat), "p_value": float(p_val), "stars": stars}


qa_metrics = QAMetrics()
retrieval_metrics = RetrievalMetrics()
stat_tests = StatisticalTests()
formatter = PromptFormatter()
context_builder = ContextBuilder()
print("✓ Métriques et utilitaires initialisés")

## Cellule 13 — Protocole A : Ablation end-to-end

In [ ]:
import datetime
import jsonlines

FACTORIAL_PLAN = [
    {"retriever": "bm25",   "k": 5,  "reranking": False, "format": "standard"},
    {"retriever": "bm25",   "k": 10, "reranking": False, "format": "standard"},
    {"retriever": "bm25",   "k": 20, "reranking": False, "format": "standard"},
    {"retriever": "dense",  "k": 5,  "reranking": False, "format": "standard"},
    {"retriever": "dense",  "k": 10, "reranking": False, "format": "standard"},
    {"retriever": "dense",  "k": 20, "reranking": False, "format": "standard"},
    {"retriever": "hybrid", "k": 5,  "reranking": False, "format": "standard"},
    {"retriever": "hybrid", "k": 10, "reranking": False, "format": "standard"},
    {"retriever": "hybrid", "k": 20, "reranking": False, "format": "standard"},
    {"retriever": "bm25",   "k": 10, "reranking": True,  "format": "standard", "initial_k": 50},
    {"retriever": "dense",  "k": 10, "reranking": True,  "format": "standard", "initial_k": 50},
    {"retriever": "bm25",   "k": 10, "reranking": False, "format": "citations"},
    {"retriever": "dense",  "k": 10, "reranking": False, "format": "citations"},
    {"retriever": "hybrid", "k": 10, "reranking": False, "format": "citations"},
    {"retriever": "hybrid", "k": 10, "reranking": True,  "format": "citations", "initial_k": 50},
]


def get_retriever(name: str, lang: str) -> object:
    """Instancie le retriever selon le nom."""
    bm25_idx = bm25_index_dirs[lang]
    if name == "bm25":
        return BM25Retriever(index_dir=bm25_idx, k1=1.0, b=0.6)
    if name == "dense":
        return DenseRetriever(index_dir=dense_index_dir, lang=lang, device=DEVICE)
    if name == "hybrid":
        bm25 = BM25Retriever(index_dir=bm25_idx)
        dense = DenseRetriever(index_dir=dense_index_dir, lang=lang, device=DEVICE)
        return HybridRetriever(bm25, dense, alpha=0.5)
    raise ValueError(f"Retriever inconnu : {name}")


def get_reranker(reranker_type: str = "colbert"):
    """Instancie le reranker ColBERT."""
    try:
        from ragatouille import RAGPretrainedModel
        model = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

        class ColBERTWrapper:
            def __init__(self, m):
                self.model = m

            def rerank(self, query: str, passages: list, top_k: int = 10) -> list:
                texts = [p.text for p in passages]
                by_text = {p.text: p for p in passages}
                results = self.model.rerank(query=query, documents=texts, k=top_k)
                reranked = []
                for rank, r in enumerate(results, start=1):
                    orig = by_text.get(r["content"])
                    if orig:
                        reranked.append(RetrievedPassage(
                            rank=rank, passage_id=orig.passage_id, text=orig.text,
                            bm25_score=orig.bm25_score, dense_score=orig.dense_score,
                            hybrid_score=orig.hybrid_score,
                            rerank_score=float(r.get("score", 0.0)),
                        ))
                return reranked[:top_k]

        return ColBERTWrapper(model)
    except Exception as e:
        logger.warning("ColBERT non disponible : %s", e)
        return None


def log_result(run_file: Path, record: dict) -> None:
    with jsonlines.open(str(run_file), mode="a") as writer:
        writer.write(record)


def make_record(question_id, question, gold_answer, config, retrieved, prompt, answer, metrics):
    return {
        "run_id": f"{config.get('protocol','X')}_{config.get('retriever','baseline')}_{config.get('lang','fr')}_{datetime.datetime.utcnow().strftime('%Y%m%d%H%M')}",
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "question_id": question_id,
        "question": question,
        "gold_answer": gold_answer,
        "config": config,
        "retrieved_passages": retrieved,
        "prompt": prompt,
        "generated_answer": answer,
        "metrics": metrics,
    }


def run_protocol_a(samples: list, lang: str, run_dir: Path) -> None:
    """Protocole A — Plan factoriel complet."""
    run_dir.mkdir(parents=True, exist_ok=True)

    # Baseline (sans retrieval)
    run_file = run_dir / f"run_baseline_{lang}.jsonl"
    config = {"retriever": None, "k": 0, "reranking": False, "format": "standard", "lang": lang, "protocol": "A"}
    for sample in tqdm(samples, desc=f"[A] Baseline {lang.upper()}"):
        prompt = formatter.format_baseline(sample["question"])
        answer = generator.generate(prompt)
        gold = sample["answers"]
        gold_list = [gold] if isinstance(gold, str) else gold
        metrics = {
            "em": qa_metrics.exact_match(answer, gold_list, lang),
            "f1": qa_metrics.token_f1(answer, gold_list, lang),
        }
        log_result(run_file, make_record(sample["id"], sample["question"], gold_list, config, [], prompt, answer, metrics))

    # Configurations factorielles
    for exp_cfg in FACTORIAL_PLAN:
        cfg_name = f"{exp_cfg['retriever']}_k{exp_cfg['k']}{'_rerank' if exp_cfg.get('reranking') else ''}_{exp_cfg['format']}"
        run_file = run_dir / f"run_{cfg_name}_{lang}.jsonl"
        full_cfg = {**exp_cfg, "lang": lang, "protocol": "A", "generator": "google/mt5-large"}

        retriever = get_retriever(exp_cfg["retriever"], lang)
        reranker = get_reranker("colbert") if exp_cfg.get("reranking") else None

        for sample in tqdm(samples, desc=f"[A] {cfg_name} {lang.upper()}"):
            initial_k = exp_cfg.get("initial_k", exp_cfg["k"])
            passages = retriever.retrieve(sample["question"], k=initial_k)
            if reranker and passages:
                passages = reranker.rerank(sample["question"], passages, top_k=exp_cfg["k"])
            passages = passages[:exp_cfg["k"]]
            texts = [p.text for p in passages]
            prompt = formatter.format(sample["question"], texts, fmt=exp_cfg["format"])
            answer = generator.generate(prompt)
            gold = sample["answers"]
            gold_list = [gold] if isinstance(gold, str) else gold
            metrics = {
                "em": qa_metrics.exact_match(answer, gold_list, lang),
                "f1": qa_metrics.token_f1(answer, gold_list, lang),
            }
            log_result(run_file, make_record(
                sample["id"], sample["question"], gold_list, full_cfg,
                [p.to_dict() for p in passages], prompt, answer, metrics
            ))


print("✓ Protocole A défini")

## Cellule 14 — Protocole B : Bruit de contexte contrôlé

In [ ]:
def run_protocol_b(samples: list, lang: str, run_dir: Path) -> None:
    """
    Protocole B — Bruit de contexte contrôlé.
    Teste H1 (seuil de bruit) et H3 (effet de position).
    """
    run_dir.mkdir(parents=True, exist_ok=True)

    NOISE_RATIOS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    POSITIONS = ["first", "middle", "last"]
    K_TOTAL = 10

    retriever = get_retriever("dense", lang)

    # Pré-charger un pool de passages distracteurs (top-50 de la première question)
    distractor_pool = retriever.retrieve(samples[0]["question"], k=50) if samples else []

    for noise_ratio in NOISE_RATIOS:
        for position in POSITIONS:
            cfg_name = f"noise{int(noise_ratio*100)}_pos{position}"
            run_file = run_dir / f"run_{cfg_name}_{lang}.jsonl"
            config = {
                "retriever": "dense", "k": K_TOTAL,
                "noise_ratio": noise_ratio, "position": position,
                "lang": lang, "protocol": "B",
            }

            for sample in tqdm(samples, desc=f"[B] {cfg_name} {lang.upper()}"):
                # Passage gold = premier passage dense pour ce sample
                gold_passages = retriever.retrieve(sample["question"], k=1)
                if not gold_passages:
                    continue
                gold = gold_passages[0]
                distractors = [p for p in distractor_pool if p.passage_id != gold.passage_id]

                ctx = context_builder.build_with_distractors(
                    gold, distractors, k_total=K_TOTAL,
                    distractor_ratio=noise_ratio, position=position
                )

                texts = [p.text for p in ctx]
                prompt = formatter.format_standard(sample["question"], texts)
                answer = generator.generate(prompt)
                gold_list = sample["answers"]
                if isinstance(gold_list, str):
                    gold_list = [gold_list]

                metrics = {
                    "em": qa_metrics.exact_match(answer, gold_list, lang),
                    "f1": qa_metrics.token_f1(answer, gold_list, lang),
                    "noise_ratio": noise_ratio,
                    "position": position,
                }
                log_result(run_file, make_record(
                    sample["id"], sample["question"], gold_list, config,
                    [p.to_dict() for p in ctx], prompt, answer, metrics
                ))


print("✓ Protocole B défini")

## Cellule 15 — Protocole C : Factualité et attribution

In [ ]:
def run_protocol_c(samples: list, lang: str, run_dir: Path) -> None:
    """
    Protocole C — Factualité et attribution.
    Teste H2 (reranking) et H3 (construction du contexte).
    """
    run_dir.mkdir(parents=True, exist_ok=True)

    CONFIGS_C = [
        {"name": "rag_standard",  "retriever": "dense", "k": 10, "reranking": False, "format": "standard"},
        {"name": "rag_rerank",    "retriever": "dense", "k": 10, "reranking": True,  "format": "standard", "initial_k": 50},
        {"name": "rag_citations", "retriever": "dense", "k": 10, "reranking": False, "format": "citations"},
        {"name": "rag_verify",    "retriever": "dense", "k": 10, "reranking": True,  "format": "citations", "initial_k": 50},
    ]

    for exp_cfg in CONFIGS_C:
        run_file = run_dir / f"run_{exp_cfg['name']}_{lang}.jsonl"
        full_cfg = {**exp_cfg, "lang": lang, "protocol": "C"}

        retriever = get_retriever(exp_cfg["retriever"], lang)
        reranker = get_reranker("colbert") if exp_cfg.get("reranking") else None

        for sample in tqdm(samples, desc=f"[C] {exp_cfg['name']} {lang.upper()}"):
            initial_k = exp_cfg.get("initial_k", exp_cfg["k"])
            passages = retriever.retrieve(sample["question"], k=initial_k)
            if reranker and passages:
                passages = reranker.rerank(sample["question"], passages, top_k=exp_cfg["k"])
            passages = passages[:exp_cfg["k"]]
            texts = [p.text for p in passages]
            prompt = formatter.format(sample["question"], texts, fmt=exp_cfg["format"])
            answer = generator.generate(prompt)
            gold = sample["answers"]
            gold_list = [gold] if isinstance(gold, str) else gold

            metrics = {
                "em": qa_metrics.exact_match(answer, gold_list, lang),
                "f1": qa_metrics.token_f1(answer, gold_list, lang),
            }
            log_result(run_file, make_record(
                sample["id"], sample["question"], gold_list, full_cfg,
                [p.to_dict() for p in passages], prompt, answer, metrics
            ))


print("✓ Protocole C défini")

## Cellule 16 — Lancement des expériences
> ⚠️ Cette cellule lance les protocoles choisis. Sur Kaggle T4, compter ~1h pour 200 samples × 15 configs.

In [ ]:
protocols = ["A", "B", "C"] if PROTOCOL == "all" else [PROTOCOL]
run_base = REPO_DIR

for protocol in protocols:
    for lang in langs:
        samples = all_samples.get(lang, [])
        logger.info("=== Protocole %s — %s — %d samples ===", protocol, lang.upper(), len(samples))
        print(f"\n{'='*60}")
        print(f"Protocole {protocol} | Langue {lang.upper()} | {len(samples)} samples")
        print("="*60)

        if protocol == "A":
            run_protocol_a(samples, lang, run_base / "runs/A_ablation")
        elif protocol == "B":
            run_protocol_b(samples, lang, run_base / "runs/B_context_noise")
        elif protocol == "C":
            run_protocol_c(samples, lang, run_base / "runs/C_factuality")

        logger.info("✓ Protocole %s — %s terminé", protocol, lang.upper())
        print(f"✓ Protocole {protocol} — {lang.upper()} terminé")

print("\n" + "="*60)
print("Toutes les expériences terminées.")
print("="*60)

## Cellule 17 — Analyse des résultats

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120


def load_run_results(run_dir: Path) -> pd.DataFrame:
    """Charge tous les fichiers JSONL d'un répertoire en DataFrame."""
    records = []
    for f in sorted(run_dir.glob("run_*.jsonl")):
        try:
            with jsonlines.open(str(f)) as reader:
                for rec in reader:
                    cfg = rec.get("config", {})
                    metrics = rec.get("metrics", {})
                    records.append({
                        "run_file": f.name,
                        "retriever": cfg.get("retriever", "baseline"),
                        "k": cfg.get("k", 0),
                        "reranking": cfg.get("reranking", False),
                        "format": cfg.get("format", "standard"),
                        "lang": cfg.get("lang", "fr"),
                        "protocol": cfg.get("protocol", "A"),
                        "noise_ratio": cfg.get("noise_ratio", None),
                        "position": cfg.get("position", None),
                        "em": metrics.get("em", 0),
                        "f1": metrics.get("f1", 0),
                        "question": rec.get("question", ""),
                        "generated_answer": rec.get("generated_answer", ""),
                    })
        except Exception as e:
            logger.warning("Erreur lecture %s : %s", f.name, e)
    return pd.DataFrame(records)


# ── Charger les résultats ─────────────────────────────────────────────────────
dfs = {}
for protocol, subdir in [("A", "A_ablation"), ("B", "B_context_noise"), ("C", "C_factuality")]:
    run_dir = REPO_DIR / "runs" / subdir
    if run_dir.exists() and any(run_dir.glob("*.jsonl")):
        dfs[protocol] = load_run_results(run_dir)
        print(f"Protocole {protocol} : {len(dfs[protocol])} enregistrements")
    else:
        print(f"Protocole {protocol} : aucun résultat (non exécuté)")


# ── Résultats Protocole A ─────────────────────────────────────────────────────
if "A" in dfs:
    df_a = dfs["A"]
    summary_a = df_a.groupby(["retriever", "k", "reranking", "format", "lang"])[["em", "f1"]].mean().reset_index()
    summary_a = summary_a.sort_values("f1", ascending=False)
    print("\n=== Protocole A — Top configurations (F1) ===")
    print(summary_a.head(10).to_string(index=False))

    # Sauvegarde CSV
    summary_a.to_csv(str(REPO_DIR / "reports/tables/protocol_a_summary.csv"), index=False)
    print("\n✓ Tableau sauvegardé → reports/tables/protocol_a_summary.csv")

## Cellule 18 — Visualisations

In [ ]:
figures_dir = REPO_DIR / "reports/figures"
figures_dir.mkdir(exist_ok=True)

if "A" in dfs and not dfs["A"].empty:
    df_a = dfs["A"]

    # Figure 1 : F1 par retriever et k
    fig, axes = plt.subplots(1, len(langs), figsize=(8 * len(langs), 5))
    if len(langs) == 1:
        axes = [axes]

    for ax, lang in zip(axes, langs):
        df_lang = df_a[(df_a["lang"] == lang) & (~df_a["reranking"]) & (df_a["format"] == "standard") & (df_a["retriever"].notna())]
        for retriever in ["bm25", "dense", "hybrid"]:
            sub = df_lang[df_lang["retriever"] == retriever].groupby("k")["f1"].mean().reset_index()
            if not sub.empty:
                ax.plot(sub["k"], sub["f1"], marker="o", label=retriever.upper())
        ax.set_title(f"F1 vs k — {lang.upper()}")
        ax.set_xlabel("k (nombre de passages)")
        ax.set_ylabel("F1 moyen")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(figures_dir / "fig1_f1_vs_k.png"), bbox_inches="tight")
    plt.show()

if "B" in dfs and not dfs["B"].empty:
    df_b = dfs["B"]

    # Figure 2 : Impact du bruit sur F1
    fig, ax = plt.subplots(figsize=(8, 5))
    for position in ["first", "middle", "last"]:
        sub = df_b[df_b["position"] == position].groupby("noise_ratio")["f1"].mean().reset_index()
        if not sub.empty:
            ax.plot(sub["noise_ratio"], sub["f1"], marker="o", label=f"Gold {position}")
    ax.set_title("F1 vs ratio de bruit (Protocole B)")
    ax.set_xlabel("Ratio de distracteurs")
    ax.set_ylabel("F1 moyen")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(figures_dir / "fig2_noise_impact.png"), bbox_inches="tight")
    plt.show()

print("✓ Figures sauvegardées dans", figures_dir)

## Cellule 19 — Tests statistiques

In [ ]:
if "A" in dfs and not dfs["A"].empty:
    df_a = dfs["A"]

    # Comparer BM25 vs Dense (k=10, standard, sans reranking)
    bm25_scores = df_a[(df_a["retriever"] == "bm25") & (df_a["k"] == 10) & (~df_a["reranking"]) & (df_a["format"] == "standard")]["f1"].tolist()
    dense_scores = df_a[(df_a["retriever"] == "dense") & (df_a["k"] == 10) & (~df_a["reranking"]) & (df_a["format"] == "standard")]["f1"].tolist()

    if bm25_scores and dense_scores:
        n = min(len(bm25_scores), len(dense_scores))
        a, b = bm25_scores[:n], dense_scores[:n]

        perm = stat_tests.permutation_test(a, b, n_permutations=1000)
        ci_bm25 = stat_tests.bootstrap_ci(a)
        ci_dense = stat_tests.bootstrap_ci(b)

        print(f"BM25   F1 : {ci_bm25['mean']:.3f} IC95[{ci_bm25['ci_lower']:.3f}, {ci_bm25['ci_upper']:.3f}]")
        print(f"Dense  F1 : {ci_dense['mean']:.3f} IC95[{ci_dense['ci_lower']:.3f}, {ci_dense['ci_upper']:.3f}]")
        print(f"Test permutation : diff={perm['observed_diff']:.3f}, p={perm['p_value']:.3f} {perm['stars']}")

        try:
            wilcox = stat_tests.wilcoxon_test(a, b)
            print(f"Wilcoxon : p={wilcox['p_value']:.3f} {wilcox['stars']}")
        except Exception as e:
            print(f"Wilcoxon non disponible : {e}")
    else:
        print("Pas assez de données pour les tests statistiques")
else:
    print("Protocole A non exécuté — pas de tests statistiques")

## Cellule 20 — Sauvegarde et archivage des résultats

In [ ]:
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
archive_name = f"/kaggle/working/rag_results_{timestamp}.zip"

dirs_to_archive = [
    REPO_DIR / "runs",
    REPO_DIR / "reports",
]

with zipfile.ZipFile(archive_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in dirs_to_archive:
        if d.exists():
            for fpath in d.rglob("*"):
                if fpath.is_file():
                    arcname = fpath.relative_to(REPO_DIR)
                    zf.write(fpath, arcname)

import os
size_mb = os.path.getsize(archive_name) / 1e6
print(f"✓ Archive créée : {archive_name} ({size_mb:.1f} MB)")
print("Télécharger depuis l'onglet Output de Kaggle.")